> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production note (active)

The FastAPI backend is already library-integrated. Routes in `apps/api/routers/` import from `mrta.*`:

```python
from mrta.core.rag_pipeline import rag_query
from mrta.core.llm import LLMClient
from mrta.retrieval.vector_store import VectorStore
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 05.

# Phase 5 — FastAPI Backend

**Goal:** Wrap the RAG pipeline from Phase 4 into a clean HTTP API with three endpoints:

- `POST /upload`   — upload a PDF, parse and index it.
- `POST /ask`      — ask a question; return answer + citations.
- `GET  /documents` — list indexed documents.

Plus `/health` for readiness probes and `/docs` for the auto-generated OpenAPI UI.

This is where the project starts looking production-shaped. We use Pydantic v2 for typed request/response models, dependency injection for shared resources, and `httpx` to call the live server from this notebook.


## 5.1 — Why FastAPI?

- Type-driven (Pydantic v2) — request/response validation is free.
- Async-friendly — useful for streaming LLM responses later.
- Auto OpenAPI + Swagger UI — every endpoint gets a doc page at `/docs`.
- Production-grade ASGI server (`uvicorn`) — supports workers, hot-reload, graceful shutdown.

Alternatives considered: Flask (no native async or types), Starlette (lower level), Litestar (newer, smaller ecosystem).


## 5.2 — Pydantic request / response schemas


In [ ]:
# Full implementation: see apps/api/schemas/

## 5.3 — The FastAPI app

Here is the file that lives at `apps/api/main.py`. We use FastAPI's lifespan event to load the embedder, vector store, and LLM exactly once at startup.


In [ ]:
# Full implementation: see apps/api/main.py

## 5.4 — Running it locally

```bash
# From the repo root, with the venv activated:
uvicorn apps.api.main:app --reload --port 8000
```

Then visit:

- http://localhost:8000/docs — Swagger UI
- http://localhost:8000/health — should return `{"status": "ok"}`


## 5.5 — Driving the API from this notebook

Once the server is running in another terminal, the rest of this notebook talks to it over HTTP. This is exactly how the Streamlit UI in Phase 6 will talk to the backend, so it is good practice.


In [ ]:
import httpx

BASE = "http://localhost:8000"

def alive() -> bool:
    try:
        r = httpx.get(f"{BASE}/health", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

if alive():
    print("API is up.")
else:
    print("API is not running. Start it in another terminal:")
    print("  uvicorn app.api.main:app --reload --port 8000")


## 5.6 — Upload a PDF


In [ ]:
from pathlib import Path

pdf_path = Path("data/raw/attention_is_all_you_need.pdf")
if not pdf_path.exists():
    raise FileNotFoundError("Run Notebook 01 first to download the sample PDF.")

if alive():
    with pdf_path.open("rb") as f:
        files = {"file": (pdf_path.name, f, "application/pdf")}
        r = httpx.post(f"{BASE}/upload", files=files, timeout=120)
    r.raise_for_status()
    print(r.json())
else:
    print("Skipping — server not running.")


## 5.7 — Ask a question


In [ ]:
if alive():
    payload = {"question": "What is multi-head attention and why is it useful?", "k": 5}
    r = httpx.post(f"{BASE}/ask", json=payload, timeout=120)
    r.raise_for_status()
    resp = r.json()
    print("Q:", resp["question"])
    print("\nA:", resp["answer"])
    print("\ncited:", resp["cited_pages"], " | latency:", round(resp["latency_s"], 2), "s")


## 5.8 — List documents


In [ ]:
if alive():
    r = httpx.get(f"{BASE}/documents")
    r.raise_for_status()
    for d in r.json():
        print(f"  {d['doc_id']:30s}  {d['source']:40s}  pages={d['n_pages']}  chunks={d['n_chunks']}")


## 5.9 — Production-shaped touches

Things to add when you graduate from "works on my machine":

| Concern         | Recommended addition                                                            |
|-----------------|----------------------------------------------------------------------------------|
| Auth            | `fastapi.security.APIKeyHeader` + a `Depends(verify_key)` on write endpoints.    |
| Rate limiting   | `slowapi` middleware, per-IP buckets.                                            |
| Tracing         | `opentelemetry-instrumentation-fastapi`; export to Jaeger or LangSmith.          |
| Errors          | Global exception handler that returns JSON, not HTML.                            |
| Validation      | Use `pydantic.Field(...)` ranges (we did); reject empty PDFs at upload.          |
| Streaming       | Switch `/ask` to `StreamingResponse` and stream the LLM tokens.                  |
| Background jobs | For large PDFs, return `202 Accepted` + a job_id; index in a Celery worker.      |

We will add Docker + logging in Notebook 09. Streaming is a great post-tutorial exercise.


## What you learned

- The full set of endpoints needed for a RAG app: `upload`, `ask`, `documents`, `health`.
- How to use Pydantic v2 for request/response validation.
- The lifespan pattern for loading expensive resources once.
- Driving the API from Python via `httpx`.
- A roadmap from "works on my laptop" to production-shaped.

## Exercises

1. Add a `DELETE /documents/{doc_id}` endpoint that removes a doc + its chunks (needs a vector store that supports filtering, e.g. Qdrant).
2. Convert `/ask` to a streaming endpoint that yields tokens as the LLM produces them.
3. Add a `/ask` rate limit of 30 requests/minute per IP.

## Next: [Phase 6 — Streamlit frontend](./2026-05-25-phase06-streamlit-frontend.ipynb)
